# 动手练习：复杂系统与网络的数学表示

来自《生物医学网络科学》（Biomedical Network Science）。

每个任务先运行“准备”格，再在“你的代码”格中完成 TODO，最后运行“核对”格：打印 ✓ 表示通过，否则会给出提示。

## 任务 1：哥尼斯堡七桥与握手定理

4 块陆地 A、B、C、D 由 7 座桥相连。两块陆地之间可能不止一座桥，所以用多重图 `MultiGraph` 表示。

1. 求每个节点的度，存为字典 `deg`；
2. 统计奇数度节点的个数 `n_odd`，按 Euler 判据（奇数度节点为 0 或 2 个）判断是否存在“每座桥恰好走一次”的路线，结果存为布尔值 `has_path`；
3. 在 B 和 C 之间再加一座桥，重新计算 `n_odd_new` 和 `has_path_new`。

期望看到：度为 A=5、B=3、C=3、D=3，度之和 14 = 2 × 7（握手定理）；原图不存在欧拉路径，加桥后存在。

> **提示：** `sum(1 for _, d in G.degree() if d % 2 == 1)`；加桥用 `G2 = G.copy(); G2.add_edge("B", "C")`。

In [ ]:
# 准备（直接运行）
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt

bridges = [("A", "B"), ("A", "B"), ("A", "C"), ("A", "C"), ("A", "D"), ("B", "D"), ("C", "D")]
G = nx.MultiGraph()
G.add_edges_from(bridges)

In [ ]:
deg = None          # TODO: dict(G.degree())
n_odd = None        # TODO: 奇数度节点个数
has_path = None     # TODO: 按 Euler 判据判断（True / False）

G2 = G.copy()
# TODO: 在 B 与 C 之间加一座桥
n_odd_new = None
has_path_new = None

In [ ]:
# 核对（不用修改）
assert deg is not None and n_odd is not None and has_path is not None, "deg、n_odd、has_path 还没有赋值"
assert deg == {"A": 5, "B": 3, "C": 3, "D": 3}, "度应为 A=5, B=3, C=3, D=3；检查是否用了 MultiGraph（普通 Graph 会把重复的桥合并）"
assert sum(deg.values()) == 2 * len(bridges), "度之和应等于 2 × 桥数（握手定理）"
assert n_odd == 4, "七桥图有 4 个奇数度节点"
assert has_path is False or has_path == False, "4 个奇数度节点时不存在欧拉路径"
assert has_path == nx.has_eulerian_path(G), "你的判断与 nx.has_eulerian_path 不一致"
assert n_odd_new == 2, "加一座 B–C 桥后，B、C 的度变为 4，只剩 A、D 两个奇数度节点；检查是否在 G2 上加了桥"
assert bool(has_path_new) and nx.has_eulerian_path(G2), "奇数度节点为 2 个时存在欧拉路径"
print("✓ 任务 1 通过")

## 任务 2：从边列表到邻接矩阵

下面是一个用于练习的小型蛋白质相互作用网络（边列表）。把它转换成邻接矩阵 `A`（numpy 整数数组，行列顺序按 `nodes`），并计算：

- 度向量 `k`（每行之和）；
- 网络密度 `rho` $= M / [N(N-1)/2]$。

然后验证三条性质：`A` 对称；对角线全为 0；度之和 = 2 × 边数。

> **提示：** 先 `A = np.zeros((N, N), dtype=int)`，再对每条边 `(u, v)` 设 `A[idx[u], idx[v]] = A[idx[v], idx[u]] = 1`。

In [ ]:
# 准备（直接运行）
edges = [("TP53", "MDM2"), ("TP53", "EP300"), ("MDM2", "MDM4"), ("TP53", "MDM4"),
         ("EP300", "CREBBP"), ("TP53", "CREBBP"), ("ATM", "TP53"), ("ATM", "CHEK2"),
         ("CHEK2", "TP53")]
nodes = sorted({u for e in edges for u in e})
idx = {name: i for i, name in enumerate(nodes)}
N, M = len(nodes), len(edges)

In [ ]:
A = None      # TODO: N×N 邻接矩阵
k = None      # TODO: 度向量（行和）
rho = None    # TODO: 密度

In [ ]:
# 核对（不用修改）
assert A is not None and k is not None and rho is not None, "A、k、rho 还没有赋值"
A = np.asarray(A)
assert A.shape == (N, N), "A 应为 %d×%d 矩阵" % (N, N)
assert np.array_equal(A, A.T), "无向网络的邻接矩阵应对称：每条边要同时设置 A[i,j] 和 A[j,i]"
assert np.all(np.diag(A) == 0), "简单图的对角线应全为 0"
assert np.array_equal(A, nx.to_numpy_array(nx.Graph(edges), nodelist=nodes, dtype=int)), "A 与 NetworkX 的结果不一致，检查行列顺序是否按 nodes"
assert np.array_equal(np.asarray(k), A.sum(axis=1)), "度应为每一行之和"
assert int(np.sum(k)) == 2 * M, "度之和应等于 2M = %d" % (2 * M)
assert abs(rho - 9 / 21) < 1e-6, "密度应为 M/(N(N-1)/2) = 9/21 ≈ 0.429；检查分母是否用了 N(N-1)/2"
print("✓ 任务 2 通过")

## 任务 3：有向网络：入度、出度与矩阵约定

一个用于练习的小型基因调控网络：边 X → Y 表示转录因子 X 调控基因 Y。本站约定 $A_{ij}=1$ 表示 $i \to j$（行是起点），这也是 `nx.to_numpy_array` 的约定。

1. 求入度字典 `in_deg`、出度字典 `out_deg`；
2. 找出出度最大的“主调控因子” `master`；
3. 得到有向邻接矩阵 `A_dir`（行列顺序按 `genes`），验证它不对称，且**行和 = 出度、列和 = 入度**。

思考：g3 → TF3 与 TF3 → g3 构成了什么结构？

> **提示：** `dict(D.in_degree())`；`max(out_deg, key=out_deg.get)`。

In [ ]:
# 准备（直接运行）
reg = [("TF1", "TF2"), ("TF1", "TF3"), ("TF1", "g1"), ("TF2", "g1"), ("TF2", "g2"),
       ("TF3", "g2"), ("TF3", "g3"), ("TF1", "g4"), ("g3", "TF3")]
D = nx.DiGraph(reg)
genes = sorted(D.nodes())

In [ ]:
in_deg = None    # TODO
out_deg = None   # TODO
master = None    # TODO: 出度最大的节点
A_dir = None     # TODO: nx.to_numpy_array(D, nodelist=genes, dtype=int)

In [ ]:
# 核对（不用修改）
assert in_deg is not None and out_deg is not None and master is not None and A_dir is not None, "还有变量没有赋值"
assert out_deg["TF1"] == 4 and in_deg["g1"] == 2 and in_deg["TF3"] == 2, "入度/出度不对：TF1 调控 4 个靶点；g1 被 TF1、TF2 调控；TF3 被 TF1、g3 调控"
assert sum(in_deg.values()) == sum(out_deg.values()) == len(reg), "有向图中入度之和 = 出度之和 = 边数"
assert master == "TF1", "出度最大的是 TF1；检查 max 用的是 out_deg 而不是 in_deg"
A_dir = np.asarray(A_dir)
assert not np.array_equal(A_dir, A_dir.T), "有向网络的邻接矩阵一般不对称"
assert [int(x) for x in A_dir.sum(axis=1)] == [out_deg[g] for g in genes], "按本站约定，行和应等于出度；检查是否把矩阵转置了"
assert [int(x) for x in A_dir.sum(axis=0)] == [in_deg[g] for g in genes], "按本站约定，列和应等于入度"
print("✓ 任务 3 通过")

## 任务 4：矩阵乘方与游走计数

$(A^r)_{ij}$ 等于从 $i$ 到 $j$ 的长度为 $r$ 的游走条数。以专题页的例子（蛋白 a、b、c、d，边 a–b、a–c、b–c、c–d）为例：

1. 计算 `A2` $=A^2$、`A3` $=A^3$（用 `@` 或 `np.linalg.matrix_power`，不要用 `A**2`，那是逐元素平方）；
2. 读出 a 与 d 的共同邻居数 `common_ad` $=(A^2)_{ad}$；
3. 用 $\operatorname{tr}(A^3)/6$ 求三角形个数 `n_tri`；
4. 对任务 2 的 TP53 网络（矩阵 `A_p53` 已给出）同样求三角形个数 `n_tri_p53`。

期望看到：$A^2$ 的对角元等于度；例子中只有 1 个三角形 a–b–c。

In [ ]:
# 准备（直接运行）
lab = ["a", "b", "c", "d"]
A4 = np.array([[0, 1, 1, 0],
               [1, 0, 1, 0],
               [1, 1, 0, 1],
               [0, 0, 1, 0]])
A_p53 = nx.to_numpy_array(nx.Graph(edges), nodelist=nodes, dtype=int)

In [ ]:
A2 = None          # TODO: A4 @ A4
A3 = None          # TODO
common_ad = None   # TODO: A2 中 a 行 d 列
n_tri = None       # TODO: np.trace(A3) / 6
n_tri_p53 = None   # TODO

In [ ]:
# 核对（不用修改）
assert A2 is not None and A3 is not None, "A2、A3 还没有赋值"
A2, A3 = np.asarray(A2), np.asarray(A3)
assert not np.array_equal(A2, A4 ** 2), "A**2 是逐元素平方；矩阵乘方要用 A @ A"
assert np.array_equal(A2, np.array([[2, 1, 1, 1], [1, 2, 1, 1], [1, 1, 3, 0], [1, 1, 0, 1]])), "A^2 不对，检查是否用了矩阵乘法 @"
assert np.array_equal(np.diag(A2), A4.sum(axis=1)), "A^2 的对角元应等于度"
assert np.array_equal(A3, np.array([[2, 3, 4, 1], [3, 2, 4, 1], [4, 4, 2, 3], [1, 1, 3, 0]])), "A^3 不对，应为 A^2 @ A"
assert common_ad == 1, "a 与 d 只有一个共同邻居 c，(A^2)_ad 应为 1"
assert n_tri == 1, "例子中只有一个三角形 a–b–c；tr(A^3) = 6，要除以 6"
assert n_tri_p53 == sum(nx.triangles(nx.Graph(edges)).values()) // 3 == 3, "TP53 网络有 3 个三角形（TP53 分别与 MDM2–MDM4、EP300–CREBBP、ATM–CHEK2 构成）"
print("✓ 任务 4 通过")

## 任务 5：拉普拉斯矩阵的谱

拉普拉斯矩阵 $L = D - A$，其中 $D$ 是度矩阵。仍用任务 4 的 4 节点例子：

1. 构造 `Lap`，求其特征值并从小到大排列，存为 `mu`（对称矩阵用 `np.linalg.eigvalsh`）；
2. 求归一化拉普拉斯 $\mathcal L = I - D^{-1/2} A D^{-1/2}$ 的特征值 `mu_norm`；
3. 对只有两条边 a–b、c–d 的 4 节点图，数一数拉普拉斯矩阵有几个（数值上）为 0 的特征值，存为 `n_zero`。

期望看到：`mu` 为 0、1、3、4，和为 $2M=8$；`mu_norm` 约为 0、0.771、1.5、1.729；`n_zero` 等于连通分支数 2。

> **提示：** `np.diag(A.sum(axis=1))`；数零特征值时用 `np.sum(np.abs(x) < 1e-9)`，不要用 `== 0`。

In [ ]:
# 准备（直接运行）
A_two = np.array([[0, 1, 0, 0],
                  [1, 0, 0, 0],
                  [0, 0, 0, 1],
                  [0, 0, 1, 0]])

In [ ]:
Lap = None      # TODO: D - A
mu = None       # TODO: 特征值（从小到大）
mu_norm = None  # TODO: 归一化拉普拉斯的特征值
n_zero = None   # TODO: A_two 的拉普拉斯矩阵的零特征值个数

In [ ]:
# 核对（不用修改）
assert Lap is not None and mu is not None and mu_norm is not None and n_zero is not None, "还有变量没有赋值"
Lap = np.asarray(Lap)
assert np.allclose(Lap.sum(axis=1), 0), "拉普拉斯矩阵每行之和应为 0；检查是否写成了 A - D 或忘了对角线"
assert np.all(np.diag(Lap) == A4.sum(axis=1)), "L 的对角元应为度，非对角元为 -A_ij"
assert np.allclose(np.sort(mu), [0, 1, 3, 4], atol=1e-6), "特征值应为 0、1、3、4；检查 L = D - A"
assert abs(np.sum(mu) - 8) < 1e-6, "特征值之和应等于 tr(L) = 2M = 8"
assert np.allclose(np.sort(mu_norm), [0, 0.771, 1.5, 1.729], atol=2e-3), "归一化拉普拉斯特征值应约为 0、0.771、1.5、1.729；检查 D^{-1/2} 两侧都要乘"
assert np.all((np.asarray(mu_norm) > -1e-9) & (np.asarray(mu_norm) < 2 + 1e-9)), "归一化拉普拉斯的特征值应落在 [0, 2]"
assert n_zero == 2, "零特征值的重数等于连通分支数；a–b、c–d 两个分支应得到 2（注意数值误差，用 |x| < 1e-9 判断）"
print("✓ 任务 5 通过")

## 任务 6：小分析：药物–靶点二分网络的投影

先用专题页的例子热身：3 种药物、4 个靶点的双邻接矩阵 `B_demo`，计算 `P_demo` $= BB^\top$，对角元是各药物的靶点数，非对角元是共享靶点数。

然后分析下面这个**虚构的** 5 药物 × 6 靶点网络 `B`（仅用于演示方法）：

1. 计算共享靶点数矩阵 `P` $= BB^\top$；
2. 自己选一种归一化（例如 Jaccard 系数 $|N(u)\cap N(u')|/|N(u)\cup N(u')|$），得到 5×5 的相似度矩阵 `S`，取值在 0–1 之间，对角线设为 0；
3. 找出最相似的一对药物 `best_pair`（两个药物名组成的元组），画出投影网络（边宽 = 相似度）；
4. 用一两句话写下：投影丢失了哪些信息？换一种归一化，结论会变吗？

本任务是开放的，只检查形状和取值范围。

In [ ]:
# 准备（直接运行）
B_demo = np.array([[1, 1, 0, 0],
                   [0, 1, 1, 0],
                   [0, 0, 0, 1]])
drugs = ["D1", "D2", "D3", "D4", "D5"]
targets = ["T1", "T2", "T3", "T4", "T5", "T6"]
B = np.array([[1, 1, 0, 0, 0, 0],
              [0, 1, 1, 0, 0, 0],
              [1, 1, 0, 1, 0, 0],
              [0, 0, 0, 0, 1, 1],
              [0, 0, 0, 0, 1, 1]])

In [ ]:
P_demo = None     # TODO: B_demo @ B_demo.T
P = None          # TODO
S = None          # TODO: 归一化后的相似度矩阵（5×5，0–1，对角线为 0）
best_pair = None  # TODO: 例如 ("D1", "D3")

In [ ]:
# 核对（不用修改）
assert P_demo is not None and P is not None and S is not None and best_pair is not None, "还有变量没有赋值"
assert np.array_equal(np.asarray(P_demo), [[2, 1, 0], [1, 2, 0], [0, 0, 1]]), "P_demo 应为 [[2,1,0],[1,2,0],[0,0,1]]；检查是否用了 B @ B.T（而不是 B.T @ B）"
P = np.asarray(P)
assert P.shape == (5, 5), "P 应为 药物数×药物数 = 5×5；B @ B.T 投影到药物一侧"
assert np.array_equal(np.diag(P), B.sum(axis=1)), "P 的对角元应等于各药物的靶点数"
S = np.asarray(S, dtype=float)
assert S.shape == (5, 5), "S 应为 5×5"
assert np.allclose(S, S.T), "相似度矩阵应对称"
assert np.all((S >= 0) & (S <= 1 + 1e-9)), "归一化后的相似度应在 0–1 之间"
assert np.allclose(np.diag(S), 0), "请把 S 的对角线设为 0"
assert len(best_pair) == 2 and best_pair[0] != best_pair[1] and set(best_pair) <= set(drugs), "best_pair 应是两个不同的药物名"
print("✓ 任务 6 通过")